# AgeLens — 04 External Validation: BioAge and R Survey

This notebook completes two independent checks:

1. Runs the official BioAge R package's `phenoage_calc(..., orig = TRUE)` branch on the identical AgeLens complete-case sample.
2. Recomputes survey-weighted means, age correlations, and bridging effects with R's `survey` package.

## Required files

```text
nhanes/scripts/04_bioage_survey_validation.R
nhanes/data/interim/nhanes_2015_2018_preprocessed_diagnostic.parquet
```

## BioAge version used

```text
dayoonkwon/BioAge@b1f9fc0
```

The installed function source is audited before any benchmark is accepted. The current official `orig = TRUE` branch uses the Supplement conversion constants, so the primary package-agreement target is AgeLens's `supplement` output. All outputs remain diagnostic only.


In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import glob
import json
import shutil
import subprocess

import numpy as np
import pandas as pd
from IPython.display import display

PROJECT_FOLDER_NAME = "nhanes"
INSTALL_MISSING_R_PACKAGES = True
RUN_R_AUTOMATICALLY = True

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 190)

print(f"Current working directory: {Path.cwd().resolve()}")


Current working directory: <PROJECT_ROOT>\notebooks


In [2]:
def find_project_root(folder_name: str = PROJECT_FOLDER_NAME) -> Path:
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if candidate.name.lower() == folder_name.lower():
            return candidate
    raise FileNotFoundError(f"Could not find a parent folder named '{folder_name}'.")


def find_rscript() -> Path | None:
    on_path = shutil.which("Rscript")
    if on_path:
        return Path(on_path)

    candidates: list[Path] = []
    for pattern in [
        r"C:\Program Files\R\R-*\bin\Rscript.exe",
        r"C:\Program Files\R\R-*\bin\x64\Rscript.exe",
    ]:
        candidates.extend(Path(path) for path in glob.glob(pattern))

    return sorted(candidates)[-1] if candidates else None


PROJECT_ROOT = find_project_root()
CONFIG_PATH = PROJECT_ROOT / "configs" / "agelens_config.json"
CONFIG = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

INTERIM_ROOT = PROJECT_ROOT / CONFIG["paths"]["interim_data"]
TABLES_ROOT = PROJECT_ROOT / CONFIG["paths"]["tables"]
LOGS_ROOT = PROJECT_ROOT / CONFIG["paths"]["logs"]
EXTERNAL_ROOT = PROJECT_ROOT / "data" / "external"
SCRIPTS_ROOT = PROJECT_ROOT / "scripts"

for path in [TABLES_ROOT, LOGS_ROOT, EXTERNAL_ROOT, SCRIPTS_ROOT]:
    path.mkdir(parents=True, exist_ok=True)

PREPROCESSED_PATH = INTERIM_ROOT / "nhanes_2015_2018_preprocessed_diagnostic.parquet"
R_SCRIPT_PATH = SCRIPTS_ROOT / "04_bioage_survey_validation.R"

print(f"Project root: {PROJECT_ROOT}")
print(f"R script expected at: {R_SCRIPT_PATH}")


Project root: <PROJECT_ROOT>
R script expected at: <PROJECT_ROOT>\scripts\04_bioage_survey_validation.R


## 1. Build the identical R validation input

Only harmonized complete cases are exported. Mortality variables are not used.


In [3]:
if not PREPROCESSED_PATH.exists():
    raise FileNotFoundError(
        f"Missing input: {PREPROCESSED_PATH}. Run 02_data_preprocessing.ipynb first."
    )

data = pd.read_parquet(PREPROCESSED_PATH)

required_columns = {
    "SEQN", "NHANES_CYCLE", "age_topcoded", "age_below_20",
    "WTSAF4YR", "SDMVSTRA", "SDMVPSU",
    "albumin_harmonized_g_L", "creatinine_harmonized_umol_L",
    "glucose_mmol_L", "log_crp_harmonized", "lymphocyte_percent",
    "mcv_fL", "rdw_percent", "alp_harmonized_U_L", "wbc_1000cells_uL",
    "chronological_age_years", "harmonized_phenoage_erratum_years",
    "harmonized_phenoage_supplement_years",
    "prebridge_phenoage_erratum_years",
    "prebridge_phenoage_supplement_years",
    "complete_case_harmonized", "complete_case_bridge_comparison",
}
missing = sorted(required_columns - set(data.columns))
if missing:
    raise ValueError(f"Missing required columns: {missing}")

complete = data.loc[data["complete_case_harmonized"]].copy()
if not complete["complete_case_bridge_comparison"].all():
    raise RuntimeError(
        "The harmonized complete-case sample is not identical to the bridge-comparison sample."
    )

rename_map = {
    "albumin_harmonized_g_L": "albumin_gL",
    "creatinine_harmonized_umol_L": "creat_umol",
    "glucose_mmol_L": "glucose_mmol",
    "log_crp_harmonized": "lncrp",
    "lymphocyte_percent": "lymph",
    "mcv_fL": "mcv",
    "rdw_percent": "rdw",
    "alp_harmonized_U_L": "alp",
    "wbc_1000cells_uL": "wbc",
    "chronological_age_years": "age",
    "harmonized_phenoage_erratum_years": "agelens_erratum",
    "harmonized_phenoage_supplement_years": "agelens_supplement",
    "prebridge_phenoage_erratum_years": "prebridge_erratum",
    "prebridge_phenoage_supplement_years": "prebridge_supplement",
}

export_columns = [
    "SEQN", "NHANES_CYCLE", "age_topcoded", "age_below_20",
    "WTSAF4YR", "SDMVSTRA", "SDMVPSU", *rename_map.keys(),
]
r_input = complete.loc[:, export_columns].rename(columns=rename_map)

# Write binary flags as 0/1 so R imports them consistently across locales
# and R/Python versions.
r_input["age_topcoded"] = (
    r_input["age_topcoded"].astype("int8")
)
r_input["age_below_20"] = (
    r_input["age_below_20"].astype("int8")
)

if r_input.isna().any().any():
    counts = r_input.isna().sum()
    raise RuntimeError(
        "R validation input unexpectedly contains missing values:\n"
        + counts[counts > 0].to_string()
    )

R_INPUT_PATH = TABLES_ROOT / "04_r_validation_input.csv"
r_input.to_csv(R_INPUT_PATH, index=False)

print(f"R validation input written: {R_INPUT_PATH}")
print(f"Rows: {len(r_input):,}")
display(r_input.head())


R validation input written: <PROJECT_ROOT>\results\tables\04_r_validation_input.csv
Rows: 5,223


,SEQN,NHANES_CYCLE,age_topcoded,age_below_20,WTSAF4YR,SDMVSTRA,SDMVPSU,albumin_gL,creat_umol,glucose_mmol,lncrp,lymph,mcv,rdw,alp,wbc,age,agelens_erratum,agelens_supplement,prebridge_erratum,prebridge_supplement
1,83733,2015_2016,0,0,27361.171665,125.0,1.0,43.0065,94.159702,5.6055,-1.888689,31.3,101.8,13.4,51.598482,7.3,53.0,55.183798,53.764437,54.139170,52.702604
2,83734,2015_2016,0,0,12735.546850,131.0,1.0,43.0065,100.047584,4.6620,-2.504579,29.9,90.8,14.7,50.502162,4.4,78.0,74.954364,73.860620,73.701944,72.587573
4,83736,2015_2016,0,0,19089.755435,126.0,2.0,41.0903,59.673536,4.6620,-2.617090,47.1,87.8,12.3,50.502162,4.2,42.0,27.866282,25.997007,26.358600,24.464494
5,83737,2015_2016,0,0,12900.422816,128.0,1.0,39.1741,102.570962,5.9385,-1.398711,31.7,92.6,14.1,91.048204,6.1,72.0,75.214348,74.124886,74.298358,73.193810
9,83741,2015_2016,0,0,54375.644543,128.0,2.0,42.0484,67.243670,5.2725,-1.947887,38.2,83.2,13.1,66.943802,3.5,22.0,15.372305,13.297257,14.157710,12.062658


## 2. Run BioAge and R `survey`

Place the R script at:

```text
nhanes/scripts/04_bioage_survey_validation.R
```

The first run can install missing R packages when `INSTALL_MISSING_R_PACKAGES = True`.


In [4]:
if not R_SCRIPT_PATH.exists():
    raise FileNotFoundError(
        f"R script is missing: {R_SCRIPT_PATH}"
    )

rscript = find_rscript()
print(f"Rscript: {rscript}")

R_STDOUT_LOG = LOGS_ROOT / "04_r_stdout.log"
R_STDERR_LOG = LOGS_ROOT / "04_r_stderr.log"
R_ERROR_LOG = LOGS_ROOT / "04_r_external_validation_error.txt"

if rscript is None:
    print(
        "Rscript was not found. Install R and rerun this notebook. "
        "The R input file has already been created."
    )
elif RUN_R_AUTOMATICALLY:
    command = [
        str(rscript),
        str(R_SCRIPT_PATH),
        str(PROJECT_ROOT),
        str(INSTALL_MISSING_R_PACKAGES).lower(),
    ]

    completed = subprocess.run(
        command,
        cwd=PROJECT_ROOT,
        text=True,
        capture_output=True,
    )

    R_STDOUT_LOG.write_text(
        completed.stdout or "",
        encoding="utf-8",
        errors="replace",
    )
    R_STDERR_LOG.write_text(
        completed.stderr or "",
        encoding="utf-8",
        errors="replace",
    )

    stdout_lines = (completed.stdout or "").splitlines()
    stderr_lines = (completed.stderr or "").splitlines()

    if stdout_lines:
        print("\n".join(stdout_lines[-200:]))

    if stderr_lines:
        print("\n--- R stderr: last 200 lines ---")
        print("\n".join(stderr_lines[-200:]))

    if R_ERROR_LOG.exists():
        error_lines = R_ERROR_LOG.read_text(
            encoding="utf-8",
            errors="replace",
        ).splitlines()
        if error_lines:
            print("\n--- R diagnostic error log ---")
            print("\n".join(error_lines[-200:]))

    if completed.returncode != 0:
        raise RuntimeError(
            "R external validation failed with exit code "
            f"{completed.returncode}. Full output was saved to:\n"
            f"  {R_STDOUT_LOG}\n"
            f"  {R_STDERR_LOG}\n"
            f"  {R_ERROR_LOG}"
        )

    print("✅ R external validation completed.")


Rscript: C:\Program Files\R\R-4.5.1\bin\x64\Rscript.exe

--- R stderr: last 200 lines ---
UyarÄ± mesajlarÄ±:
1: package 'survey' was built under R version 4.5.3 
2: package 'dplyr' was built under R version 4.5.3 
BioAge stable benchmark and R survey validation completed successfully. Raw package non-finite outputs are documented in 04_bioage_numerical_stability_audit.csv.
✅ R external validation completed.


## 3. BioAge agreement

The official package output is written to the benchmark file used by `03_validation.ipynb`.


In [5]:
BIOAGE_COMPARISON_PATH = TABLES_ROOT / "04_bioage_comparison.csv"
BENCHMARK_PATH = EXTERNAL_ROOT / "bioage_phenoage_benchmark.csv"
SOURCE_AUDIT_PATH = TABLES_ROOT / "04_bioage_source_token_audit.csv"
STABILITY_AUDIT_PATH = (
    TABLES_ROOT / "04_bioage_numerical_stability_audit.csv"
)

if not BIOAGE_COMPARISON_PATH.exists():
    print("BioAge comparison output is not present yet.")
else:
    bioage_comparison = pd.read_csv(BIOAGE_COMPARISON_PATH)
    source_audit = pd.read_csv(SOURCE_AUDIT_PATH)
    stability_audit = pd.read_csv(STABILITY_AUDIT_PATH)
    benchmark = pd.read_csv(BENCHMARK_PATH)

    if not source_audit["present"].all():
        raise RuntimeError("BioAge source-token audit did not pass.")

    if len(benchmark) != len(r_input):
        raise RuntimeError(
            "BioAge benchmark row count does not match the identical input."
        )

    finite_metric_columns = [
        "mae",
        "rmse",
        "mean_agelens_minus_bioage",
        "sd_difference",
        "bland_altman_lower",
        "bland_altman_upper",
        "pearson",
        "spearman",
    ]

    if not np.isfinite(
        bioage_comparison[finite_metric_columns].to_numpy(dtype=float)
    ).all():
        raise RuntimeError(
            "BioAge comparison still contains non-finite metrics."
        )

    if not stability_audit["stable_finite_n"].eq(
        stability_audit["n"]
    ).all():
        raise RuntimeError(
            "Stable BioAge benchmark contains non-finite values."
        )

    display(source_audit)
    display(stability_audit)
    display(bioage_comparison.round(10))

    if "finite_subset_within_1e8_tolerance" in stability_audit.columns:
        tolerance_status = stability_audit[
            "finite_subset_within_1e8_tolerance"
        ]
        print(
            "Raw-package finite-subset 1e-8 tolerance status:",
            tolerance_status.tolist(),
        )

    supplement_rows = bioage_comparison.loc[
        bioage_comparison["variant"].eq("supplement")
    ]

    print(
        "Stable exact-BioAge supplement MAE range:",
        float(supplement_rows["mae"].min()),
        "to",
        float(supplement_rows["mae"].max()),
        "years",
    )

    raw_nonfinite = int(
        stability_audit["package_nonfinite_n"].sum()
    )
    print(
        "Raw BioAge package non-finite values documented:",
        raw_nonfinite,
    )


,token,present
0,-19.906670,True
1,-0.033594,True
2,0.009506,True
3,0.195319,True
4,0.095368,True
5,-0.012000,True
6,0.026764,True
7,0.330616,True
8,0.001869,True
9,0.055424,True


,NHANES_CYCLE,n,package_finite_n,package_nonfinite_n,package_positive_infinity_n,package_negative_infinity_n,stable_finite_n,max_abs_package_minus_stable_finite_subset,finite_subset_within_1e8_tolerance
0,2015_2016,2645,2642,3,3,0,2645,7.897333e-08,False
1,2017_2018,2578,2575,3,3,0,2578,3.211450e-05,False


,NHANES_CYCLE,variant,n,mae,rmse,mean_agelens_minus_bioage,sd_difference,bland_altman_lower,bland_altman_upper,pearson,spearman,difference_average_correlation
0,2015_2016,erratum,2645,1.638000,1.679885,1.637691,0.374212,0.904236,2.371146,1.0,1.0,-0.999766
1,2015_2016,supplement,2645,0.049726,0.051009,0.049726,0.011372,0.027437,0.072014,1.0,1.0,0.696215
2,2017_2018,erratum,2578,1.601316,1.644821,1.600600,0.378911,0.857934,2.343265,1.0,1.0,-0.999788
3,2017_2018,supplement,2578,0.050348,0.051643,0.050337,0.011546,0.027706,0.072967,1.0,1.0,0.731621


Raw-package finite-subset 1e-8 tolerance status: [False, False]
Stable exact-BioAge supplement MAE range: 0.0497257249378904 to 0.0503475696870256 years
Raw BioAge package non-finite values documented: 6


## 4. Python versus R survey cross-check

In [6]:
PYTHON_MEANS_PATH = TABLES_ROOT / "03_survey_weighted_summaries.csv"
R_MEANS_PATH = TABLES_ROOT / "04_r_survey_weighted_means.csv"
PYTHON_CORR_PATH = TABLES_ROOT / "03_age_correlations.csv"
R_CORR_PATH = TABLES_ROOT / "04_r_survey_age_correlations.csv"
PYTHON_BRIDGE_PATH = TABLES_ROOT / "03_bridge_validation.csv"
R_BRIDGE_PATH = TABLES_ROOT / "04_r_bridge_validation.csv"

survey_crosscheck = pd.DataFrame()
correlation_crosscheck = pd.DataFrame()
bridge_crosscheck = pd.DataFrame()

if R_MEANS_PATH.exists():
    py = pd.read_csv(PYTHON_MEANS_PATH)
    rr = pd.read_csv(R_MEANS_PATH)
    survey_crosscheck = py.merge(
        rr, on=["sample", "cycle", "formula_variant"],
        suffixes=("_python", "_r"), validate="one_to_one"
    )
    survey_crosscheck["mean_difference_python_minus_r"] = (
        survey_crosscheck["weighted_mean_python"] - survey_crosscheck["weighted_mean_r"]
    )
    survey_crosscheck["se_difference_python_minus_r"] = (
        survey_crosscheck["taylor_se_python"] - survey_crosscheck["taylor_se_r"]
    )
    display(survey_crosscheck[[
        "sample", "cycle", "formula_variant",
        "weighted_mean_python", "weighted_mean_r", "mean_difference_python_minus_r",
        "taylor_se_python", "taylor_se_r", "se_difference_python_minus_r",
    ]].round(12))
    print(
        "Maximum absolute mean difference:",
        float(survey_crosscheck["mean_difference_python_minus_r"].abs().max())
    )
    print(
        "Maximum absolute SE difference:",
        float(survey_crosscheck["se_difference_python_minus_r"].abs().max())
    )

if R_CORR_PATH.exists():
    py = pd.read_csv(PYTHON_CORR_PATH)
    rr = pd.read_csv(R_CORR_PATH)
    correlation_crosscheck = py.merge(
        rr, on=["sample", "cycle", "formula_variant", "n"], validate="one_to_one"
    )
    correlation_crosscheck["difference_python_minus_r"] = (
        correlation_crosscheck["pearson_weighted_point_estimate"]
        - correlation_crosscheck["weighted_pearson"]
    )
    display(correlation_crosscheck[[
        "sample", "cycle", "formula_variant",
        "pearson_weighted_point_estimate", "weighted_pearson",
        "difference_python_minus_r",
    ]].round(12))

if R_BRIDGE_PATH.exists():
    py = pd.read_csv(PYTHON_BRIDGE_PATH)
    rr = pd.read_csv(R_BRIDGE_PATH)
    bridge_crosscheck = py.merge(
        rr, on=["cycle", "formula_variant", "n_identical_sample"],
        suffixes=("_python", "_r"), validate="one_to_one"
    )
    bridge_crosscheck["mean_difference_python_minus_r"] = (
        bridge_crosscheck["mean_post_minus_pre_weighted_python"]
        - bridge_crosscheck["mean_post_minus_pre_weighted_r"]
    )
    bridge_crosscheck["se_difference_python_minus_r"] = (
        bridge_crosscheck["difference_taylor_se_python"]
        - bridge_crosscheck["difference_taylor_se_r"]
    )
    display(bridge_crosscheck.round(12))


,sample,cycle,formula_variant,weighted_mean_python,weighted_mean_r,mean_difference_python_minus_r,taylor_se_python,taylor_se_r,se_difference_python_minus_r
0,all_harmonized_complete_case,2015_2016,erratum,43.748566,43.748566,0.0,0.816922,0.816922,0.0
1,all_harmonized_complete_case,2015_2016,supplement,42.140869,42.140869,0.0,0.830376,0.830376,0.0
2,all_harmonized_complete_case,2017_2018,erratum,44.816034,44.816034,0.0,0.649945,0.649945,-0.0
3,all_harmonized_complete_case,2017_2018,supplement,43.225918,43.225918,-0.0,0.660650,0.660650,0.0
4,no_topcode,2015_2016,erratum,42.435602,42.435602,0.0,0.767708,0.767708,-0.0
5,no_topcode,2015_2016,supplement,40.806280,40.806280,0.0,0.780352,0.780352,-0.0
6,no_topcode,2017_2018,erratum,43.231941,43.231941,0.0,0.574818,0.574818,-0.0
7,no_topcode,2017_2018,supplement,41.615735,41.615735,0.0,0.584285,0.584285,-0.0
8,age20plus,2015_2016,erratum,47.996484,47.996484,0.0,0.821895,0.821895,0.0
9,age20plus,2015_2016,supplement,46.458749,46.458749,-0.0,0.835432,0.835432,-0.0


Maximum absolute mean difference: 4.973799150320701e-14
Maximum absolute SE difference: 4.440892098500626e-16


,sample,cycle,formula_variant,pearson_weighted_point_estimate,weighted_pearson,difference_python_minus_r
0,all_harmonized_complete_case,2015_2016,erratum,0.943354,0.943354,0.0
1,all_harmonized_complete_case,2015_2016,supplement,0.943354,0.943354,0.0
2,all_harmonized_complete_case,2017_2018,erratum,0.945125,0.945125,-0.0
3,all_harmonized_complete_case,2017_2018,supplement,0.945125,0.945125,-0.0
4,no_topcode,2015_2016,erratum,0.938958,0.938958,-0.0
5,no_topcode,2015_2016,supplement,0.938958,0.938958,-0.0
6,no_topcode,2017_2018,erratum,0.938986,0.938986,0.0
7,no_topcode,2017_2018,supplement,0.938986,0.938986,0.0
8,age20plus,2015_2016,erratum,0.925556,0.925556,0.0
9,age20plus,2015_2016,supplement,0.925556,0.925556,0.0


,cycle,formula_variant,n_identical_sample,pre_weighted_mean,post_weighted_mean,mean_post_minus_pre_weighted_python,difference_taylor_se_python,difference_ci_low_95_python,difference_ci_high_95_python,weighted_sd_post_minus_pre,mean_post_minus_pre_weighted_r,difference_taylor_se_r,difference_ci_low_95_r,difference_ci_high_95_r,mean_difference_python_minus_r,se_difference_python_minus_r
0,2015_2016,erratum,2645,42.405280,43.748566,1.343286,0.014397,1.315067,1.371505,0.538275,1.343286,0.014397,1.315068,1.371504,-0.0,0.0
1,2015_2016,supplement,2645,40.775460,42.140869,1.365410,0.014635,1.336726,1.394093,0.547140,1.365410,0.014635,1.336726,1.394093,0.0,-0.0
2,2017_2018,erratum,2578,44.816034,44.816034,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0
3,2017_2018,supplement,2578,43.225918,43.225918,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.0


## 5. Save cross-check artifacts

In [7]:
written = []
for filename, frame in {
    "04_python_r_survey_mean_crosscheck.csv": survey_crosscheck,
    "04_python_r_correlation_crosscheck.csv": correlation_crosscheck,
    "04_python_r_bridge_crosscheck.csv": bridge_crosscheck,
}.items():
    if not frame.empty:
        path = TABLES_ROOT / filename
        frame.to_csv(path, index=False)
        written.append(path)

metadata = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "notebook": "04_external_validation.ipynb",
    "rscript_found": rscript is not None,
    "r_validation_completed": BIOAGE_COMPARISON_PATH.exists(),
    "bioage_reference": "dayoonkwon/BioAge@b1f9fc0",
    "bioage_expected_primary_match": "supplement",
    "mortality_data_used": False,
    "outputs_diagnostic_only": True,
    "open_core_evidence_gaps": sorted(CONFIG["governance"]["open_core_evidence_gaps"]),
    "written_crosscheck_files": [str(path.relative_to(PROJECT_ROOT)) for path in written],
}
metadata_path = LOGS_ROOT / "04_external_validation_metadata.json"
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")
print(f"Metadata written: {metadata_path}")


Metadata written: <PROJECT_ROOT>\logs\04_external_validation_metadata.json


## 6. Final status

In [8]:
if BIOAGE_COMPARISON_PATH.exists():
    print("✅ Official BioAge orig=TRUE comparison completed.")
    print("✅ R survey package comparison completed.")
    print("Rerun 03_validation.ipynb to generate its populated BioAge comparison tables.")
else:
    print("R validation has not run yet. Install R and rerun this notebook.")

print("All outputs remain diagnostic only.")


✅ Official BioAge orig=TRUE comparison completed.
✅ R survey package comparison completed.
Rerun 03_validation.ipynb to generate its populated BioAge comparison tables.
All outputs remain diagnostic only.
